#  Quote Guessing Game
**Web-scrape every quote from [quotes.toscrape.com](http://quotes.toscrape.com), then challenge yourself to guess the author.**

---
### How it works
1. **Scrape** — All pages are collected and saved to a CSV for reuse.
2. **Play** — A random quote is displayed; you have **3 guesses**.
3. **Hints** — Each wrong answer unlocks a progressively easier clue:
   - After 1st wrong guess → birthdate & birthplace
   - After 2nd wrong guess → author's first-name initial
   - After 3rd wrong guess → author's last-name initial

---
### Bug fixes applied to the original code
| # | Bug | Fix |
|---|-----|-----|
| 1 | `soup.find(_class="next")` — invalid kwarg, pagination silently broke | Changed to `class_="next"` |
| 2 | `base_url` ended with `/` and `url` started with `/` — double slash | Removed trailing slash from `BASE_URL` |
| 3 | `remaining_guesses = 2` but hints fired at 3 / 2 / 1 — bio hint unreachable | Set `MAX_GUESSES = 3` |
| 4 | `if guess == quote["author"]` — case-sensitive check inside a `.lower()` loop | Made all comparisons consistently case-insensitive |
| 5 | Game-over message was in an `else` branch that never ran correctly | Moved message outside the loop |

## 1 · Imports

In [ ]:
import requests
from bs4 import BeautifulSoup
from csv import writer, reader
from time import sleep
from random import choice
from pathlib import Path
from IPython.display import clear_output, display, HTML

## 2 · Configuration

In [ ]:
# FIX 2: no trailing slash — avoids double-slash in concatenated URLs
BASE_URL     = "http://quotes.toscrape.com"
CSV_FILE     = Path("quotes.csv")
MAX_GUESSES  = 3   # FIX 3: was 2; hints are coded for values 3 → 2 → 1
SLEEP_SECS   = 1   # polite delay between page requests

## 3 · Scraper

Scrapes every page until there is no **Next** button, then caches results to `quotes.csv`.
On subsequent runs the CSV is loaded instantly — no re-scraping needed.

In [ ]:
def scrape_all_quotes() -> list[dict]:
    """Scrape every quote from quotes.toscrape.com page by page."""
    all_quotes = []
    url = "/page/1/"

    while url:
        full_url = f"{BASE_URL}{url}"
        try:
            res = requests.get(full_url, timeout=10)
            res.raise_for_status()
        except requests.RequestException as e:
            print(f"   Request failed: {e}")
            break

        print(f"  Scraping {full_url}")
        soup = BeautifulSoup(res.text, "html.parser")

        for q in soup.find_all(class_="quote"):
            all_quotes.append({
                "text":     q.find(class_="text").get_text(),
                "author":   q.find(class_="author").get_text(),
                "bio-link": q.find("a")["href"],
            })

        # FIX 1: was `_class=` (invalid keyword) — pagination never advanced
        next_btn = soup.find(class_="next")
        url = next_btn.find("a")["href"] if next_btn else None
        sleep(SLEEP_SECS)

    return all_quotes


def save_quotes_csv(quotes: list[dict]) -> None:
    """Write quotes to CSV_FILE."""
    with CSV_FILE.open("w", newline="", encoding="utf-8") as f:
        w = writer(f)
        w.writerow(["text", "author", "bio-link"])
        for q in quotes:
            w.writerow([q["text"], q["author"], q["bio-link"]])


def load_quotes_csv() -> list[dict]:
    """Load quotes from CSV_FILE."""
    with CSV_FILE.open(encoding="utf-8") as f:
        rows = list(reader(f))
    headers = rows[0]
    return [dict(zip(headers, row)) for row in rows[1:]]

In [ ]:
if CSV_FILE.exists():
    print(f" Loading quotes from cache ({CSV_FILE}) …")
    all_quotes = load_quotes_csv()
else:
    print(" No cache found — scraping all pages …")
    all_quotes = scrape_all_quotes()
    save_quotes_csv(all_quotes)
    print(f" Saved to {CSV_FILE}")

print(f"\n {len(all_quotes)} quotes ready.")

## 4 · Helper Functions

In [ ]:
def fetch_bio_hint(bio_link: str) -> str:
    """Fetch birthdate and birthplace from the author's bio page."""
    try:
        res = requests.get(f"{BASE_URL}{bio_link}", timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        date  = soup.find(class_="author-born-date").get_text()
        place = soup.find(class_="author-born-location").get_text()
        return f"The author was born on {date} {place}."
    except (requests.RequestException, AttributeError):
        return "(Bio hint unavailable)"


def get_hint(remaining: int, quote: dict) -> str:
    """Return the appropriate hint string for the current remaining-guess count."""
    # FIX 3 cont.: thresholds now correctly map to MAX_GUESSES = 3
    if remaining == 2:
        return f" Hint: {fetch_bio_hint(quote['bio-link'])}"
    elif remaining == 1:
        return f" Hint: The author's first name starts with '{quote['author'][0]}'"
    else:  # remaining == 0, shown before the game-over message
        last_initial = quote["author"].split()[1][0]
        return f" Hint: The author's last name starts with '{last_initial}'"


def show_quote_card(quote_text: str) -> None:
    """Render the quote in a styled HTML card inside the notebook."""
    html = f"""
    <div style="
        border-left: 5px solid #6c63ff;
        background: #f9f9ff;
        padding: 18px 24px;
        border-radius: 8px;
        font-family: Georgia, serif;
        font-size: 1.15em;
        color: #333;
        max-width: 680px;
        margin: 12px 0;
        line-height: 1.6;
    ">
        {quote_text}
    </div>
    """
    display(HTML(html))


def show_feedback(msg: str, colour: str = "#333") -> None:
    """Render a coloured feedback line."""
    display(HTML(f'<p style="font-size:1em;color:{colour};margin:6px 0">{msg}</p>'))

## 5 · Game

Run the cell below to start a round. Re-run it to play again with a new quote.

In [ ]:
# ── Pick a quote ──────────────────────────────────────────────────────────────
quote     = choice(all_quotes)
remaining = MAX_GUESSES
won       = False

display(HTML('<h3 style="font-family:sans-serif"> Guess the author of this quote:</h3>'))
show_quote_card(quote["text"])

# ── Guess loop ────────────────────────────────────────────────────────────────
while remaining > 0:
    guess = input(f"Your guess [{remaining} guess{'es' if remaining != 1 else ''} left]: ").strip()

    if not guess:
        show_feedback("Please enter a name.", "#e67e22")
        continue

    # FIX 4: consistent case-insensitive comparison throughout
    if guess.lower() == quote["author"].lower():
        show_feedback(f" Correct! The author is <strong>{quote['author']}</strong>.", "#27ae60")
        won = True
        break

    remaining -= 1
    show_feedback("✗ Not quite.", "#e74c3c")

    if remaining > 0:
        show_feedback(get_hint(remaining, quote), "#8e44ad")
    else:
        # Last hint (last-name initial) before revealing the answer
        show_feedback(get_hint(remaining, quote), "#8e44ad")

# FIX 5: game-over message is outside the loop — always reached when remaining == 0
if not won:
    show_feedback(
        f" Out of guesses! The answer was <strong>{quote['author']}</strong>.",
        "#c0392b"
    )

## 6 · Bonus — Explore the Data
Quick stats and a peek at the scraped dataset.

In [ ]:
from collections import Counter

author_counts = Counter(q["author"] for q in all_quotes)
top_authors   = author_counts.most_common(5)

print(f"Total quotes  : {len(all_quotes)}")
print(f"Unique authors: {len(author_counts)}")
print("\nTop 5 most-quoted authors:")
for rank, (author, count) in enumerate(top_authors, 1):
    print(f"  {rank}. {author} — {count} quote{'s' if count != 1 else ''}")

In [ ]:
# Preview the first 5 scraped quotes
print(f"{'AUTHOR':<25} {'QUOTE (first 70 chars)'}")
print("-" * 80)
for q in all_quotes[:5]:
    short_text = q["text"][:70].rstrip() + ("…" if len(q["text"]) > 70 else "")
    print(f"{q['author']:<25} {short_text}")